# CarbonGlobe v3 — extrapolation-validated residual models

**What the leaderboard told us (v2):** anchor + 0.5·residual scored 0.068, anchor + 1.0·residual scored 0.089, while random/cluster CV said the opposite.
So CV was too optimistic about how much of the learned residual transfers to SSP climates, and the fix has to be in validation and robustness, not in more capacity.

**Changes**
1. Test y0 is mapped with *random tie-breaking* (v2 collapsed tied sites onto one value, which also made the adversarial check meaningless). Per-variable adversarial AUC and age-correlation gaps are printed; variables that still look different after mapping are dropped from the model inputs (height/agb are always kept for the anchor).
2. **Directional extrapolation CV**: hold out the sites that lie furthest along the global→test climate shift direction (and the opposite end), with the hold-out size chosen so the shift matches the real one. Hold-outs are also scored with noisy y0. A random fold is printed for reference only.
3. Residual weights per target and age group are fitted by NNLS on those extrapolation folds, then capped (SHRINK_CAP).
4. Lower-capacity models (depth 6, min_child_weight 100, lambda 30) because rows from one site are correlated.
5. Three climate treatments: M1 = y0 only; M2 = shift-filtered climate (as v2); M3 = *all* climate features after pooled mean/std alignment of the test set to the global set.
6. Four submission files that double as experiments (see the last cell).

In [1]:
import os, time, gc, warnings
import numpy as np, pandas as pd, xgboost as xgb
from scipy.optimize import nnls
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

TRAIN_DIR = os.environ.get("TRAIN_DIR", "/kaggle/input/datasets/zhihaow/carbonglobe")
COMP_DIR = os.environ.get("COMP_DIR", "/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public")
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working")
QUICK = int(os.environ.get("QUICK", "0"))

G = os.path.join(TRAIN_DIR, "data_global")
AGES = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
AG = np.array(AGES, np.float64); NA = 15
SCEN = ["ssp126", "ssp585"]

SAFE_SHIFT, MIN_CLIM_FEATS, K_CLIM = 0.30, 80, 80
NOISE, EVAL_NOISE = 0.08, 0.10       # y0 noise: training augmentation / hold-out stress test (in y_std units)
SHRINK_CAP = 0.70                    # max total residual weight; v2 leaderboard: 0.5 -> 0.068, 1.0 -> 0.089
AUC_DROP = 0.85                      # y0 variables (other than height/agb) whose mapped AUC exceeds this are not used as model inputs
FINAL_SEEDS = [0, 1]
ETA, DEPTH, MCW, MAX_ROUNDS, ES_ROUNDS = 0.06, 6, 100, 2500, 60
if QUICK: ETA, MAX_ROUNDS, ES_ROUNDS, FINAL_SEEDS = 0.3, 40, 15, [0]

try: DEVICE = "cuda" if xgb.build_info().get("USE_CUDA") else "cpu"
except Exception: DEVICE = "cpu"
t0 = time.time()
def tick(m): print(f"[{(time.time()-t0)/60:5.1f} min] {m}", flush=True)
print("device:", DEVICE, "| xgboost", xgb.__version__)

# what is in the data folders (look for extra covariates we are not using yet)
for p in [G, os.path.join(TRAIN_DIR, "data_stats"), COMP_DIR]:
    try: print(p, "->", sorted(os.listdir(p))[:40])
    except Exception as e: print(p, "unreadable:", e)
for sc in SCEN:
    z = np.load(os.path.join(COMP_DIR, f"test_{sc}.npz")); print(sc, {k: z[k].shape for k in z.files})

st = np.load(os.path.join(TRAIN_DIR, "data_stats", "data_stats.npz"))
SCALE = np.array([float(st["y_mean"][0]), float(st["y_mean"][1])]); Y_STD = st["y_std"].astype(np.float32)
NS = np.load(f"{G}/glob_Y001.npy", mmap_mode="r").shape[0]

device: cuda | xgboost 3.4.1
/kaggle/input/datasets/zhihaow/carbonglobe/data_global -> ['glob_X_fea.npy', 'glob_Y001.npy', 'glob_Y010.npy', 'glob_Y020.npy', 'glob_Y030.npy', 'glob_Y050.npy', 'glob_Y070.npy', 'glob_Y100.npy', 'glob_Y150.npy', 'glob_Y200.npy', 'glob_Y250.npy', 'glob_Y300.npy', 'glob_Y350.npy', 'glob_Y400.npy', 'glob_Y450.npy', 'glob_Y500.npy', 'res_train4_test8.npz']
/kaggle/input/datasets/zhihaow/carbonglobe/data_stats -> ['age_triplet.npz', 'data_stats.npz', 'mask2d.npy']
/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public -> ['README.md', 'sample_submission.csv', 'sites_ssp.csv', 'test_ssp126.npz', 'test_ssp585.npz']
ssp126 {'test_x': (6171, 40, 12, 136), 'test_y0': (6171, 15, 7)}
ssp585 {'test_x': (6171, 40, 12, 136), 'test_y0': (6171, 15, 7)}


## 1. Global reference data

In [2]:
CACHE_T = os.path.join(OUT_DIR, "glob_targets_v3.npz")
if os.path.exists(CACHE_T):
    z = np.load(CACHE_T); Y0g, T2g = z["Y0g"], z["T2g"]
else:
    Y0g = np.zeros((NS, NA, 7), np.float32); T2g = np.zeros((NS, NA, 2), np.float32)
    for ai, a in enumerate(AGES):
        g = np.load(f"{G}/glob_Y{a:03d}.npy", mmap_mode="r")
        Y0g[:, ai] = np.asarray(g[:, 0, -1, :]); T2g[:, ai] = np.asarray(g[:, -1, -1, :2]) / SCALE
    np.savez(CACHE_T, Y0g=Y0g, T2g=T2g)
tick(f"global targets ready: {NS} sites")
SSPY = {sc: np.load(os.path.join(COMP_DIR, f"test_{sc}.npz"))["test_y0"].astype(np.float32) for sc in SCEN}
print("ssp126 y0 identical to ssp585 y0:", np.allclose(SSPY["ssp126"], SSPY["ssp585"]))

[  9.6 min] global targets ready: 54152 sites
ssp126 y0 identical to ssp585 y0: False


## 2. Map test y0 onto the global distribution (random tie-breaking) and check what is still different

In [3]:
ref_sorted = np.sort(Y0g, axis=0); QGRID = np.linspace(0, 1, NS)
def qmap(Y, seed=0):
    rng = np.random.default_rng(seed); n = len(Y); out = np.empty_like(Y); q = (np.arange(n) + 0.5) / n
    for k in range(7):
        for ai in range(NA):
            v = Y[:, ai, k]; order = np.lexsort((rng.random(n), v)); r = np.empty(n); r[order] = q
            out[:, ai, k] = np.interp(r, QGRID, ref_sorted[:, ai, k])
    return out
SSPM = {sc: qmap(Y) for sc, Y in SSPY.items()}

def adv(Xg, Xts, rounds=100, seed=0):
    """AUC of a classifier telling global sites (0) from test sites (1); held-out part split by site so SSP copies never straddle."""
    rng = np.random.default_rng(seed); nt = max(len(x) for x in Xts); flag = rng.random(nt) < 0.3
    Xb = np.vstack(Xts); teb = flag[np.concatenate([np.arange(len(x)) for x in Xts])]
    ng = min(len(Xg), 6 * len(Xb)); ia = rng.choice(len(Xg), ng, replace=False); tea = rng.random(ng) < 0.3
    X = np.vstack([Xg[ia], Xb]); y = np.r_[np.zeros(ng), np.ones(len(Xb))]; te = np.r_[tea, teb]
    p = dict(objective="binary:logistic", eval_metric="auc", eta=0.1, max_depth=4, subsample=0.8, colsample_bytree=0.8, tree_method="hist", device=DEVICE, seed=seed)
    b = xgb.train(p, xgb.DMatrix(X[~te], label=y[~te]), rounds)
    return float(roc_auc_score(y[te], b.predict(xgb.DMatrix(X[te]))))

rows = []
for k in range(7):
    cg = np.nan_to_num(np.corrcoef(Y0g[:, :, k].T)); cm = np.nan_to_num(np.corrcoef(SSPM["ssp126"][:, :, k].T))
    rows.append(dict(var=k, auc_raw=adv(Y0g[:, :, k], [SSPY["ssp126"][:, :, k]]), auc_mapped=adv(Y0g[:, :, k], [SSPM["ssp126"][:, :, k]]),
                     age_corr_gap=float(np.abs(cg - cm).mean())))
dg = pd.DataFrame(rows); print(dg.round(3).to_string(index=False))
Y0_VARS = [k for k in range(7) if k < 2 or dg.auc_mapped[k] <= AUC_DROP]
print("y0 variables used as model inputs:", Y0_VARS, "(0.5 AUC = mapped test looks like global; age_corr_gap ~0 = same age-to-age structure)")
print("floor of raw test y0 vs global (age 1 / age 150), per variable:")
for k in range(7): print(f"  v{k}: global min {Y0g[:,0,k].min():.3f}/{Y0g[:,7,k].min():.3f} | test min {SSPY['ssp585'][:,0,k].min():.3f}/{SSPY['ssp585'][:,7,k].min():.3f}")
tick("y0 mapping diagnostics done")

 var  auc_raw  auc_mapped  age_corr_gap
   0    1.000       0.955         0.018
   1    1.000       0.831         0.006
   2    1.000       0.873         0.029
   3    0.991       0.854         0.011
   4    0.996       0.797         0.003
   5    0.996       0.861         0.003
   6    0.996       0.681         0.006
y0 variables used as model inputs: [0, 1, 4, 6] (0.5 AUC = mapped test looks like global; age_corr_gap ~0 = same age-to-age structure)
floor of raw test y0 vs global (age 1 / age 150), per variable:
  v0: global min 1.500/0.000 | test min 0.415/-0.901
  v1: global min 0.000/0.000 | test min -0.399/-0.401
  v2: global min 0.011/0.000 | test min -0.523/-0.532
  v3: global min 0.000/0.000 | test min 0.119/0.118
  v4: global min -0.005/-0.126 | test min 0.249/0.254
  v5: global min -0.008/-0.146 | test min 0.117/0.041
  v6: global min 0.000/0.000 | test min 0.085/0.085
[  9.7 min] y0 mapping diagnostics done


## 3. Climate summaries (8 statistics per driver, cached — same cache name as v2)

In [ ]:
S_NAMES = ["mean", "sd_year", "sd_month", "late-early", "season_amp", "top3_months", "bot3_months", "late10_mean"]
def clim_feats(x, chunk=500):
    n, nd = len(x), x.shape[-1]; out = np.zeros((n, len(S_NAMES), nd), np.float32)
    for i in range(0, n, chunk):
        c = np.asarray(x[i:i + chunk], dtype=np.float32); A = c.mean(2); Cm = np.sort(c.mean(1), axis=1)
        out[i:i + chunk] = np.stack([A.mean(1), A.std(1), c.std(2).mean(1), A[:, -10:].mean(1) - A[:, :10].mean(1),
                                     Cm[:, -1] - Cm[:, 0], Cm[:, -3:].mean(1), Cm[:, :3].mean(1), A[:, -10:].mean(1)], 1)
    return np.nan_to_num(out)
CACHE = os.path.join(OUT_DIR, "clim_feats_v3.npz")
if os.path.exists(CACHE):
    z = np.load(CACHE); CG, CS = z["CG"], {sc: z[f"S_{sc}"] for sc in SCEN}
else:
    CG = clim_feats(np.load(f"{G}/glob_X_fea.npy", mmap_mode="r")); tick("global climate")
    CS = {sc: clim_feats(np.load(os.path.join(COMP_DIR, f"test_{sc}.npz"))["test_x"]) for sc in SCEN}
    np.savez(CACHE, CG=CG, **{f"S_{sc}": CS[sc] for sc in SCEN})
ND = CG.shape[2]; CGa = CG.reshape(NS, -1); CSa = {sc: CS[sc].reshape(len(CS[sc]), -1) for sc in SCEN}
fname = lambda j: f"{S_NAMES[j // ND]}:d{j % ND}"
tick(f"climate features ready: {CGa.shape[1]} candidates")

## 4. Two ways to deal with the climate shift
* **M2 inputs**: keep only features whose test mean is within `SAFE_SHIFT` global stds (drop the rest).
* **M3 inputs**: keep *all* non-degenerate features, but align the pooled test set (both SSPs together, so their difference survives) to the global mean/std per feature.

In [ ]:
mu, sd = CGa.mean(0), CGa.std(0); nd_ok = sd > 1e-6
shift = np.max([np.abs(CSa[sc].mean(0) - mu) / (sd + 1e-9) for sc in SCEN], axis=0)
for thr in (SAFE_SHIFT, 0.45, 0.6, 0.8, 1.2):
    cand2 = np.where(nd_ok & (shift < thr))[0]
    if len(cand2) >= MIN_CLIM_FEATS: break
cand3 = np.where(nd_ok)[0]
P = np.vstack([CSa[sc] for sc in SCEN]); mP, sP = P.mean(0), P.std(0); ratio = np.clip(sd / (sP + 1e-9), 0.5, 2.0); del P
CS3 = {sc: ((CSa[sc] - mP) * ratio + mu).astype(np.float32) for sc in SCEN}
print(f"M2 candidates (shift<{thr}): {len(cand2)} | M3 candidates (all non-degenerate): {len(cand3)}")
print("climate adversarial AUC (global vs test): M2 features raw %.3f | M3 features raw %.3f | M3 features aligned %.3f" % (
    adv(CGa[:, cand2], [CSa[sc][:, cand2] for sc in SCEN]), adv(CGa[:, cand3], [CSa[sc][:, cand3] for sc in SCEN]),
    adv(CGa[:, cand3], [CS3[sc][:, cand3] for sc in SCEN])))
tick("climate treatments ready")

## 5. Age anchor

In [ ]:
def interp_curve(y, q):
    q = np.clip(q, AG[0], AG[-1]); hi = np.clip(np.searchsorted(AG, q, side="left"), 1, NA - 1); lo = hi - 1
    w = ((q - AG[lo]) / (AG[hi] - AG[lo]))[None, :, None]
    return y[:, lo, :] * (1 - w) + y[:, hi, :] * w
coef = np.zeros((NA, 2, 2)); c40g = interp_curve(Y0g[..., :2] / SCALE, AG + 40)
for a in range(NA):
    for k in range(2):
        coef[a, k] = np.linalg.lstsq(np.stack([c40g[:, a, k], np.ones(NS)], 1), T2g[:, a, k], rcond=None)[0]
def anchor(Y7):
    c = interp_curve(Y7[..., :2] / SCALE, AG + 40)
    return np.stack([coef[None, :, k, 0] * c[..., k] + coef[None, :, k, 1] for k in range(2)], -1)
Rsite = T2g - anchor(Y0g)
print("anchor-only MSE on global sites:", float(((np.maximum(anchor(Y0g), 0) - T2g) ** 2).mean()))

## 6. Climate feature selection (site-level GBDT importance for the anchor residual)

In [ ]:
def select_features(Cf, Rs, K):
    groups = [Rs[:, sl, k].mean(1) for sl in (slice(0, 15), slice(0, 4), slice(4, 10), slice(10, 15)) for k in range(2)]
    imp = np.zeros(Cf.shape[1]); d = xgb.DMatrix(Cf)
    for g in groups:
        d.set_label(g)
        b = xgb.train(dict(objective="reg:squarederror", eta=0.1, max_depth=6, subsample=0.8, colsample_bytree=0.5, min_child_weight=20,
                           tree_method="hist", device=DEVICE, seed=0), d, 60 if QUICK else 200)
        v = np.zeros(Cf.shape[1])
        for key, val in b.get_score(importance_type="total_gain").items(): v[int(key[1:])] = val
        imp += v / (v.sum() + 1e-12)
    top = np.argsort(-imp)[:min(Cf.shape[1], 4 * K)]
    Z = (Cf[:, top] - Cf[:, top].mean(0)) / (Cf[:, top].std(0) + 1e-9); sel = []
    for t in range(len(top)):
        if len(sel) >= K: break
        if sel and np.abs(Z[:, sel].T @ Z[:, t] / len(Z)).max() > 0.97: continue
        sel.append(t)
    return top[sel]
FI = {"M2": cand2[select_features(CGa[:, cand2], Rsite, K_CLIM)], "M3": cand3[select_features(CGa[:, cand3], Rsite, K_CLIM)]}
CLG = {f: CGa[:, FI[f]] for f in FI}
CLT = {"M2": {sc: CSa[sc][:, FI["M2"]] for sc in SCEN}, "M3": {sc: CS3[sc][:, FI["M3"]] for sc in SCEN}}
for f in FI: print(f, "top 10:", [fname(j) for j in FI[f][:10]])
tick("climate selection done")

## 7. Features, models, and the directional extrapolation folds

In [ ]:
def feats(Y7, clim=None):
    n = len(Y7); a = np.broadcast_to(AG[None, :, None], (n, NA, 1)); Yv = Y7[..., Y0_VARS]; y2 = Y7[..., :2]
    c40 = interp_curve(Yv, AG + 40); c20 = interp_curve(y2, AG + 20); c60 = interp_curve(y2, AG + 60); c100 = interp_curve(y2, AG + 100)
    per = np.concatenate([Yv, c40, c20, c60, c100, c40[..., :2] - y2, c60 - c20, c100 - c40[..., :2], a, np.log(a)], -1)
    parts = [np.repeat(Yv.reshape(n, 1, -1), NA, 1), per]
    if clim is not None: parts.append(np.repeat(clim[:, None, :], NA, 1))
    return np.concatenate(parts, -1).reshape(n * NA, -1).astype(np.float32)

def noisy(Y7, seed, frac=0.5, s=NOISE):
    r = np.random.default_rng(seed); m = r.random(len(Y7)) < frac
    return Y7 + (r.normal(0, s, Y7.shape) * Y_STD * m[:, None, None]).astype(np.float32)
def build(idx, fam, Y7):
    return feats(Y7, CLG[fam][idx] if fam != "M1" else None), (T2g[idx] - anchor(Y7)).reshape(-1, 2)

PARAMS = dict(objective="reg:squarederror", eta=ETA, max_depth=DEPTH, min_child_weight=MCW, subsample=0.7, colsample_bytree=0.5,
              reg_lambda=30.0, tree_method="hist", device=DEVICE)
def fit_pair(Xtr, Rtr, Xes, Res, seed):
    dtr, des = xgb.DMatrix(Xtr), xgb.DMatrix(Xes); bs, best = [], []
    for k in range(2):
        dtr.set_label(Rtr[:, k]); des.set_label(Res[:, k])
        b = xgb.train({**PARAMS, "seed": seed}, dtr, MAX_ROUNDS, evals=[(des, "es")], early_stopping_rounds=ES_ROUNDS, verbose_eval=False)
        bs.append(b); best.append(b.best_iteration + 1)
    return bs, best
def pred_pair(bs, X, n, rounds=None):
    d = xgb.DMatrix(X)
    return np.stack([b.predict(d, iteration_range=(0, rounds[k]) if rounds else (0, 0)) for k, b in enumerate(bs)], 1).reshape(n, NA, 2)
mse = lambda A, r, T: float(((np.maximum(A + r, 0) - T) ** 2).mean())

# --- extrapolation folds: sites furthest along the global -> test climate shift direction ---
M_G = CG[:, 0, :]; M_T = np.vstack([CS[sc][:, 0, :] for sc in SCEN]); sdm = M_G.std(0); okm = sdm > 1e-6
dvec = (M_T.mean(0) - M_G.mean(0))[okm] / sdm[okm]; dn = float(np.linalg.norm(dvec)); dh = dvec / (dn + 1e-12)
u = ((M_G[:, okm] - M_G[:, okm].mean(0)) / sdm[okm]) @ dh
bestp = None
for p in np.arange(0.05, 0.401, 0.025):
    hi = u >= np.quantile(u, 1 - p); gap = u[hi].mean() - u[~hi].mean()
    if bestp is None or abs(gap - dn) < bestp[0]: bestp = (abs(gap - dn), float(p))
P_HOLD = bestp[1]
FOLDS = {"up": u >= np.quantile(u, 1 - P_HOLD), "down": u <= np.quantile(u, P_HOLD), "random": np.random.default_rng(1).random(NS) < 0.10}
print(f"global->test mean shift along its own direction: {dn:.2f} (u-units, global sd of u = {u.std():.2f}); hold-out fraction {P_HOLD:.3f}; fold sizes:", {k: int(v.sum()) for k, v in FOLDS.items()})
print("   (the 'random' fold is a reference only; it is not used for weights)")

def cv_family(fam):
    out = {}
    for name, hom in FOLDS.items():
        ho, tr = np.where(hom)[0], np.where(~hom)[0]
        es = np.random.default_rng(100).random(len(tr)) < 0.08; trf, tre = tr[~es], tr[es]
        Xtr, Rtr = build(trf, fam, noisy(Y0g[trf], 11)); Xes, Res = build(tre, fam, Y0g[tre])
        bs, best = fit_pair(Xtr, Rtr, Xes, Res, 0); del Xtr, Rtr, Xes, Res; gc.collect()
        rec = dict(best=best)
        for scen, Yh in (("clean", Y0g[ho]), ("noisy", noisy(Y0g[ho], 21, frac=1.0, s=EVAL_NOISE))):
            X, R = build(ho, fam, Yh)
            rec[scen] = dict(r=pred_pair(bs, X, len(ho), best), R=R.reshape(len(ho), NA, 2), A=anchor(Yh), T=T2g[ho])
        out[name] = rec; tick(f"  CV {fam} fold {name}: best rounds {best}")
    return out
CV = {f: cv_family(f) for f in ("M1", "M2")}

LAMS = [0, 0.25, 0.5, 0.75, 1.0]
rows = []
for fold in FOLDS:
    for scen in ("clean", "noisy"):
        c1, c2 = CV["M1"][fold][scen], CV["M2"][fold][scen]
        rows.append(dict(fold=fold, y0=scen, anchor=mse(c1["A"], 0, c1["T"]), M1=mse(c1["A"], c1["r"], c1["T"]),
                         **{f"M2@{l}": mse(c2["A"], l * c2["r"], c2["T"]) for l in LAMS[1:]}))
print(pd.DataFrame(rows).round(5).to_string(index=False))

## 8. Residual weights per target and age group (NNLS on the up/down extrapolation folds, capped)

In [ ]:
GROUPS = [slice(0, 4), slice(4, 10), slice(10, 15)]
W = np.zeros((2, 3, 2))                                        # [target, age group, model M1/M2]
for k in range(2):
    for g, sl in enumerate(GROUPS):
        A_, y_ = [], []
        for fold in ("up", "down"):
            for scen in ("clean", "noisy"):
                a1, a2 = CV["M1"][fold][scen], CV["M2"][fold][scen]
                A_.append(np.stack([a1["r"][:, sl, k].ravel(), a2["r"][:, sl, k].ravel()], 1)); y_.append(a1["R"][:, sl, k].ravel())
        W[k, g] = nnls(np.concatenate(A_).astype(np.float64), np.concatenate(y_).astype(np.float64))[0]
        s = W[k, g].sum()
        if s > SHRINK_CAP: W[k, g] *= SHRINK_CAP / s
print("weights [target][age group young/mid/old] -> (M1, M2) after cap:")
for k, nm in enumerate(["height", "agb"]): print(" ", nm, np.round(W[k], 3).tolist())
WM = np.zeros((2, NA, 2))
for g, sl in enumerate(GROUPS):
    for m in range(2): WM[m, sl, :] = W[:, g, m]
combine = lambda A, r1, r2: np.maximum(A + WM[0] * r1 + WM[1] * r2, 0)
rows = []
for fold in FOLDS:
    for scen in ("clean", "noisy"):
        c1, c2 = CV["M1"][fold][scen], CV["M2"][fold][scen]
        rows.append(dict(fold=fold, y0=scen, anchor=mse(c1["A"], 0, c1["T"]), M2_half=mse(c2["A"], 0.5 * c2["r"], c2["T"]),
                         final_blend=float(((combine(c1["A"], c1["r"], c2["r"]) - c1["T"]) ** 2).mean())))
print(pd.DataFrame(rows).round(5).to_string(index=False))

## 9. Refit on all global sites and predict the test sets

In [ ]:
FINAL = {}
for fam in ("M1", "M2", "M3"):
    src = "M2" if fam == "M3" else fam
    rounds = [int(np.mean([CV[src][f]["best"][k] for f in FOLDS]) * 1.1) + 1 for k in range(2)]; FINAL[fam] = []
    for s in FINAL_SEEDS:
        X, R = build(np.arange(NS), fam, noisy(Y0g, 1000 + s)); d = xgb.DMatrix(X); bs = []
        for k in range(2):
            d.set_label(R[:, k]); bs.append(xgb.train({**PARAMS, "seed": s}, d, rounds[k]))
        FINAL[fam].append(bs); del X, R, d; gc.collect(); tick(f"  refit {fam} seed {s} rounds {rounds}")
def predict_res(fam, Y7, clim):
    X = feats(Y7, clim); return np.mean([pred_pair(bs, X, len(Y7)) for bs in FINAL[fam]], 0)
ANCH = {sc: anchor(SSPM[sc]) for sc in SCEN}
RES = {sc: {"M1": predict_res("M1", SSPM[sc], None), "M2": predict_res("M2", SSPM[sc], CLT["M2"][sc]), "M3": predict_res("M3", SSPM[sc], CLT["M3"][sc])} for sc in SCEN}

dcv = np.mean([((CV["M2"][f]["clean"]["r"] - CV["M1"][f]["clean"]["r"]) ** 2).mean() for f in ("up", "down")])
print(f"M2-M1 mean sq. diff: extrapolation CV {dcv:.5f} | test " + " | ".join(f"{sc} {((RES[sc]['M2'] - RES[sc]['M1']) ** 2).mean():.5f}" for sc in SCEN))
print("M3-M2 mean sq. diff on test:", " | ".join(f"{sc} {((RES[sc]['M3'] - RES[sc]['M2']) ** 2).mean():.5f}" for sc in SCEN))
print("mean sq. residual on test (M1/M2/M3):", {sc: [round(float((RES[sc][m] ** 2).mean()), 4) for m in ("M1", "M2", "M3")] for sc in SCEN},
      "| mean sq. residual in CV hold-outs (truth / M2 pred):", round(float(np.mean([(CV["M2"][f]["clean"]["R"] ** 2).mean() for f in ("up", "down")])), 4),
      round(float(np.mean([(CV["M2"][f]["clean"]["r"] ** 2).mean() for f in ("up", "down")])), 4))
print("mean predicted ssp585 - ssp126 (height, agb):", {m: np.round((RES["ssp585"][m] - RES["ssp126"][m]).mean((0, 1)), 4).tolist() for m in ("M1", "M2", "M3")})

## 10. Submissions (each one is also an experiment)
* `V3_A_anchor_only.csv` — diagnostic: the leaderboard score of the anchor alone fixes the residual-weight curve (with your v2 scores 0.068 @0.5 and 0.089 @1.0 you can solve for the best weight).
* `V3_B_main.csv` — anchor + capped, CV-weighted M1/M2 residual (the default pick).
* `V3_C_clim_aligned.csv` — same weights, but the climate model M3 (all features, test aligned to global) replaces M2.
* `V3_D_M2M3_avg.csv` — average of the M2 and M3 climate residuals.

In [ ]:
sample = pd.read_csv(os.path.join(COMP_DIR, "sample_submission.csv"))
def write(preds, name):
    parts = []
    for sc, p in preds.items():
        ids = [f"{sc}_{s}_{a}" for s in range(p.shape[0]) for a in AGES]
        parts.append(pd.DataFrame({"id": ids, "height": p[..., 0].ravel(), "agb": p[..., 1].ravel()}))
    out = pd.concat(parts).set_index("id").reindex(sample["id"]); assert not out.isna().any().any() and len(out) == len(sample)
    out.reset_index().to_csv(os.path.join(OUT_DIR, name), index=False); print("saved", name)
write({sc: np.maximum(ANCH[sc], 0) for sc in SCEN}, "V3_A_anchor_only.csv")
write({sc: combine(ANCH[sc], RES[sc]["M1"], RES[sc]["M2"]) for sc in SCEN}, "V3_B_main.csv")
write({sc: combine(ANCH[sc], RES[sc]["M1"], RES[sc]["M3"]) for sc in SCEN}, "V3_C_clim_aligned.csv")
write({sc: combine(ANCH[sc], RES[sc]["M1"], 0.5 * (RES[sc]["M2"] + RES[sc]["M3"])) for sc in SCEN}, "V3_D_M2M3_avg.csv")
tick("done")

In [13]:
"""Run in any Kaggle cell after the v3 notebook (reads its CSVs; no GPU, takes seconds).
Writes three probe submissions built only from V3_A (anchor) and V3_C (anchor + residual)."""
import pandas as pd

W = "/kaggle/working/"
A = pd.read_csv(W + "V3_A_anchor_only.csv").set_index("id")
C = pd.read_csv(W + "V3_C_clim_aligned.csv").set_index("id").reindex(A.index)
age = A.index.to_series().str.split("_").str[-1].astype(int)

def save(df, name):
    assert not df.isna().any().any() and (df.values >= 0).all()
    df.reset_index().to_csv(W + name, index=False); print("saved", name)

# P1: residual weight x0.75 (effective ~0.5, where the v2 scores put the optimum)
save(A + 0.75 * (C - A), "P1_C_scale075.csv")

# P2: residual on height only, agb = anchor  -> with A and C known this fixes the height/agb split of the error
save(pd.concat([C[["height"]], A[["agb"]]], axis=1), "P2_height_resid_only.csv")

# P3: residual only for ages >= 50, young ages = anchor  -> fixes the young/old split of the error
old = (age >= 50).values
P3 = A.copy(); P3.loc[old, :] = C.loc[old, :].values
save(P3, "P3_resid_ages_ge50.csv")"""Read-only inspection. Paste into a Kaggle cell (no GPU needed) and send me the full printed output."""
import os, numpy as np, pandas as pd
TRAIN_DIR = "/kaggle/input/datasets/zhihaow/carbonglobe"
COMP_DIR = "/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public"
G = os.path.join(TRAIN_DIR, "data_global"); S = os.path.join(TRAIN_DIR, "data_stats")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 80)

print("=== README.md ===");  print(open(os.path.join(COMP_DIR, "README.md")).read()[:9000])

print("\n=== sites_ssp.csv ===")
sites = pd.read_csv(os.path.join(COMP_DIR, "sites_ssp.csv")); print(sites.shape); print(sites.head(8)); print(sites.describe(include="all").T.head(25))

def show_npz(path):
    z = np.load(path, allow_pickle=True); print(os.path.basename(path))
    for k in z.files:
        a = z[k]; print(f"  {k}: shape {a.shape} dtype {a.dtype}", "| head:", np.ravel(a)[:6].tolist() if a.size else "")
for p in [os.path.join(G, "res_train4_test8.npz"), os.path.join(S, "age_triplet.npz"), os.path.join(S, "data_stats.npz")]:
    print(); show_npz(p)
m = np.load(os.path.join(S, "mask2d.npy")); print("\nmask2d", m.shape, m.dtype, "nonzero:", int((m != 0).sum()))

print("\n=== climate drivers: global (3000-site sample) vs test ===")
def site_stats(x, chunk=500):
    Am, ch = [], []
    for i in range(0, len(x), chunk):
        a = np.asarray(x[i:i + chunk], dtype=np.float32).mean(2)            # (n, years, drivers)
        Am.append(a.mean(1)); ch.append(a[:, -10:].mean(1) - a[:, :10].mean(1))
    return np.concatenate(Am), np.concatenate(ch)
X = np.load(f"{G}/glob_X_fea.npy", mmap_mode="r"); idx = np.sort(np.random.default_rng(0).choice(X.shape[0], 3000, replace=False))
gm, gc = site_stats(X[idx])
T = {}
for sc in ["ssp126", "ssp585"]:
    T[sc] = site_stats(np.load(os.path.join(COMP_DIR, f"test_{sc}.npz"))["test_x"])
d = pd.DataFrame({"g_mean": gm.mean(0), "g_site_sd": gm.std(0), "g_change": gc.mean(0),
                  "t126_mean": T["ssp126"][0].mean(0), "t585_mean": T["ssp585"][0].mean(0),
                  "t126_change": T["ssp126"][1].mean(0), "t585_change": T["ssp585"][1].mean(0)})
d["shift_in_site_sd"] = (d.t585_mean - d.g_mean) / (d.g_site_sd + 1e-9)
d["rel_shift"] = (d.t585_mean - d.g_mean) / (d.g_mean.abs() + 1e-9)
d["site_cv"] = d.g_site_sd / (d.g_mean.abs() + 1e-9)
d["ssp_gap_rel"] = (d.t585_mean - d.t126_mean) / (d.g_mean.abs() + 1e-9)
print("top 15 by shift measured in global site-sd:");  print(d.reindex(d.shift_in_site_sd.abs().sort_values(ascending=False).index).head(15).round(4))
print("\nnearly spatially constant drivers (site_cv<0.05), largest relative shift (CO2-like):")
print(d[d.site_cv < 0.05].reindex(d[d.site_cv < 0.05].rel_shift.abs().sort_values(ascending=False).index).head(10).round(4))
print("\nlargest ssp585 vs ssp126 gap (relative to global mean):"); print(d.reindex(d.ssp_gap_rel.abs().sort_values(ascending=False).index).head(10).round(4))

saved P1_C_scale075.csv
saved P2_height_resid_only.csv
saved P3_resid_ages_ge50.csv


In [ ]:
"""Read-only inspection. Paste into a Kaggle cell (no GPU needed) and send me the full printed output."""
import os, numpy as np, pandas as pd
TRAIN_DIR = "/kaggle/input/datasets/zhihaow/carbonglobe"
COMP_DIR = "/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public"
G = os.path.join(TRAIN_DIR, "data_global"); S = os.path.join(TRAIN_DIR, "data_stats")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 80)

print("=== README.md ===");  print(open(os.path.join(COMP_DIR, "README.md")).read()[:9000])

print("\n=== sites_ssp.csv ===")
sites = pd.read_csv(os.path.join(COMP_DIR, "sites_ssp.csv")); print(sites.shape); print(sites.head(8)); print(sites.describe(include="all").T.head(25))

def show_npz(path):
    z = np.load(path, allow_pickle=True); print(os.path.basename(path))
    for k in z.files:
        a = z[k]; print(f"  {k}: shape {a.shape} dtype {a.dtype}", "| head:", np.ravel(a)[:6].tolist() if a.size else "")
for p in [os.path.join(G, "res_train4_test8.npz"), os.path.join(S, "age_triplet.npz"), os.path.join(S, "data_stats.npz")]:
    print(); show_npz(p)
m = np.load(os.path.join(S, "mask2d.npy")); print("\nmask2d", m.shape, m.dtype, "nonzero:", int((m != 0).sum()))

print("\n=== climate drivers: global (3000-site sample) vs test ===")
def site_stats(x, chunk=500):
    Am, ch = [], []
    for i in range(0, len(x), chunk):
        a = np.asarray(x[i:i + chunk], dtype=np.float32).mean(2)            # (n, years, drivers)
        Am.append(a.mean(1)); ch.append(a[:, -10:].mean(1) - a[:, :10].mean(1))
    return np.concatenate(Am), np.concatenate(ch)
X = np.load(f"{G}/glob_X_fea.npy", mmap_mode="r"); idx = np.sort(np.random.default_rng(0).choice(X.shape[0], 3000, replace=False))
gm, gc = site_stats(X[idx])
T = {}
for sc in ["ssp126", "ssp585"]:
    T[sc] = site_stats(np.load(os.path.join(COMP_DIR, f"test_{sc}.npz"))["test_x"])
d = pd.DataFrame({"g_mean": gm.mean(0), "g_site_sd": gm.std(0), "g_change": gc.mean(0),
                  "t126_mean": T["ssp126"][0].mean(0), "t585_mean": T["ssp585"][0].mean(0),
                  "t126_change": T["ssp126"][1].mean(0), "t585_change": T["ssp585"][1].mean(0)})
d["shift_in_site_sd"] = (d.t585_mean - d.g_mean) / (d.g_site_sd + 1e-9)
d["rel_shift"] = (d.t585_mean - d.g_mean) / (d.g_mean.abs() + 1e-9)
d["site_cv"] = d.g_site_sd / (d.g_mean.abs() + 1e-9)
d["ssp_gap_rel"] = (d.t585_mean - d.t126_mean) / (d.g_mean.abs() + 1e-9)
print("top 15 by shift measured in global site-sd:");  print(d.reindex(d.shift_in_site_sd.abs().sort_values(ascending=False).index).head(15).round(4))
print("\nnearly spatially constant drivers (site_cv<0.05), largest relative shift (CO2-like):")
print(d[d.site_cv < 0.05].reindex(d[d.site_cv < 0.05].rel_shift.abs().sort_values(ascending=False).index).head(10).round(4))
print("\nlargest ssp585 vs ssp126 gap (relative to global mean):"); print(d.reindex(d.ssp_gap_rel.abs().sort_values(ascending=False).index).head(10).round(4))

=== README.md ===
# CarbonGlobe Challenge - Test Data (IEEE Big Data Cup 2026)

Forecast the **future** forest carbon cycle. Train on the CarbonGlobe present-day dataset, then for
each **site** and each **seed forest age** in the two future-climate scenarios, roll your model out
**40 annual steps** from the provided initial state and report the **year-40 (final) state** of two
variables: **height** and **agb** (the long-horizon *cumulative* outcome).

## Files in this folder
| file | description |
|---|---|
| `test_ssp126.npz`, `test_ssp585.npz` | model inputs per scenario (`test_x`, `test_y0`) |
| `sites_ssp.csv` | site index for each axis-0 row (shared by both scenarios) |
| `sample_submission.csv` | the exact rows/order to submit (height & agb zero-filled) |
| `README.md` | this file |

## Scenarios (both scored; one submission covers both)
- `ssp126` - SSP1, low emissions    - `ssp585` - SSP5, high emissions
Each covers **6171 globally-distributed sites** and **15 seed ages**. A hi

In [ ]:
"""Run after the v3 notebook (reads V3_C_clim_aligned.csv). Tests the hypothesis that agb is biased low under SSP climate,
more so under ssp585 (CO2 is a driver and is far outside the global training range)."""
import pandas as pd, numpy as np
W = "/kaggle/working/"
C = pd.read_csv(W + "V3_C_clim_aligned.csv").set_index("id")
w = np.where(C.index.str.startswith("ssp585"), 1.0, 0.5)         # uplift = m * (1.0 for ssp585, 0.5 for ssp126)
for m, name in [(0.08, "U1_agb_up_0p08.csv"), (0.16, "U2_agb_up_0p16.csv")]:
    D = C.copy(); D["agb"] = C["agb"] * (1 + m * w)
    assert not D.isna().any().any(); D.reset_index().to_csv(W + name, index=False); print("saved", name)